N_TT​ = 200 r1 ​+ 30 r1 ​r2 ​+ 1401 r2​.

In [3]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorly.decomposition import tensor_train
from tensorly.tt_tensor import tt_to_tensor


# ============================================================
# PATHS
# ============================================================

LOADED_ROOT = Path("../data/preprocessed")
RESULTS_ROOT = Path("../results/tt_rank_selection")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

files = sorted(LOADED_ROOT.rglob("*.npz"))

print(f"Found {len(files)} tensor files")


# ============================================================
# TT RANKS TO TEST
# ============================================================

TT_RANKS = [
    (1, 3, 3, 1),
    (1, 3, 5, 1),
    (1, 5, 3, 1),
    (1, 5, 5, 1),
    (1, 5, 7, 1),
    (1, 7, 5, 1),
    (1, 7, 7, 1),
    (1, 7, 10, 1),
    (1, 10, 7, 1),
    (1, 10, 10, 1),
    (1, 10, 15, 1),
    (1, 15, 10, 1),
    (1, 15, 15, 1),
    (1, 15, 20, 1),
    (1, 20, 15, 1),
    (1, 20, 20, 1),
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def infer_subject_task_run(file):
    """
    Expected structure:

    data/preprocessed/002/auditory/run01_epochs.npz
    """

    parts = file.parts

    subject = file.parent.parent.name
    task = file.parent.name

    filename = file.stem

    # run01_epochs -> run01
    run = filename.split("_")[0]

    return subject, task, run

def calculate_metrics(X, X_hat, compressed_parameters):
    """
    Calculate reconstruction and compression metrics
    for an arbitrary-dimensional tensor.
    """

    X = np.asarray(X, dtype=np.float64)
    X_hat = np.asarray(X_hat, dtype=np.float64)

    # --------------------------------------------------------
    # Frobenius norm for an arbitrary-dimensional tensor
    # --------------------------------------------------------

    reconstruction_error = (
        np.linalg.norm((X - X_hat).ravel())
        /
        np.linalg.norm(X.ravel())
    )

    # --------------------------------------------------------
    # Explained variance
    # --------------------------------------------------------

    explained_variance = 1 - (
        np.sum((X - X_hat) ** 2)
        /
        np.sum(X ** 2)
    )

    # --------------------------------------------------------
    # Parameter counts
    # --------------------------------------------------------

    original_parameters = X.size

    compression_ratio = (
        1 - compressed_parameters / original_parameters
    )

    compression_factor = (
        original_parameters / compressed_parameters
    )

    return {
        "Original_Parameters": original_parameters,
        "Compressed_Parameters": compressed_parameters,
        "Compression_Ratio": compression_ratio,
        "Compression_Factor": compression_factor,
        "Reconstruction_Error": reconstruction_error,
        "Reconstruction_Error_Percent": reconstruction_error * 100,
        "Explained_Variance": explained_variance,
        "Explained_Variance_Percent": explained_variance * 100,
    }


def tt_parameter_count(shape, ranks):
    """
    Calculate number of parameters in a TT representation.

    For a 3-way tensor:

        ranks = [1, r1, r2, 1]

    Number of parameters:

        n1*r1 + n2*r1*r2 + n3*r2
    """

    n1, n2, n3 = shape
    _, r1, r2, _ = ranks

    return (
        n1 * r1
        + n2 * r1 * r2
        + n3 * r2
    )


# ============================================================
# MAIN EXPERIMENT
# ============================================================

results = []

for file_index, file in enumerate(files, start=1):

    print(
        f"\n[{file_index}/{len(files)}] "
        f"{file.parent.parent.name}/"
        f"{file.parent.name}/"
        f"{file.name}"
    )

    # --------------------------------------------------------
    # LOAD TENSOR
    # --------------------------------------------------------

    tensor = np.load(file, allow_pickle=True)

    X = tensor["epochs"].astype(np.float64)

    subject, task, run = infer_subject_task_run(file)

    print(f"Tensor shape: {X.shape}")

    # --------------------------------------------------------
    # RUN EACH TT RANK
    # --------------------------------------------------------

    for ranks in TT_RANKS:

        print(f"  TT ranks = {ranks}")

        start_time = time.perf_counter()

        try:

            # TT decomposition
            tt_cores = tensor_train(
                X,
                rank=list(ranks)
            )

            # Reconstruct tensor
            X_tt = tt_to_tensor(tt_cores)

            runtime = time.perf_counter() - start_time

            # Number of parameters
            compressed_parameters = sum(
                core.size for core in tt_cores
            )

            # Metrics
            metrics = calculate_metrics(
                X,
                X_tt,
                compressed_parameters
            )

            results.append({
                "File": str(file),
                "Subject": subject,
                "Task": task,
                "Run": run,
                "Method": "TT",
                "Rank": str(ranks),
                "Rank_R1": ranks[1],
                "Rank_R2": ranks[2],
                "Original_Parameters":
                    metrics["Original_Parameters"],
                "Compressed_Parameters":
                    metrics["Compressed_Parameters"],
                "Compression_Ratio":
                    metrics["Compression_Ratio"],
                "Compression_Factor":
                    metrics["Compression_Factor"],
                "Reconstruction_Error":
                    metrics["Reconstruction_Error"],
                "Reconstruction_Error_Percent":
                    metrics["Reconstruction_Error_Percent"],
                "Explained_Variance":
                    metrics["Explained_Variance"],
                "Explained_Variance_Percent":
                    metrics["Explained_Variance_Percent"],
                "Runtime_Seconds": runtime,
                "Status": "Success"
            })

        except Exception as e:

            print(f"    FAILED: {e}")

            results.append({
                "File": str(file),
                "Subject": subject,
                "Task": task,
                "Run": run,
                "Method": "TT",
                "Rank": str(ranks),
                "Rank_R1": ranks[1],
                "Rank_R2": ranks[2],
                "Status": "Failed",
                "Error": str(e)
            })


# ============================================================
# DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

success_df = results_df[
    results_df["Status"] == "Success"
].copy()


# ============================================================
# SAVE RAW RESULTS
# ============================================================

csv_path = RESULTS_ROOT / "tt_rank_results.csv"

results_df.to_csv(
    csv_path,
    index=False
)

print(f"\nSaved raw results to:")
print(csv_path)


# ============================================================
# OVERALL SUMMARY
# ============================================================

overall_summary = (
    success_df
    .groupby(["Rank", "Rank_R1", "Rank_R2"], as_index=False)
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error", "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error", "std"
        ),
        Mean_Reconstruction_Error_Percent=(
            "Reconstruction_Error_Percent", "mean"
        ),
        Mean_Explained_Variance=(
            "Explained_Variance", "mean"
        ),
        Mean_Explained_Variance_Percent=(
            "Explained_Variance_Percent", "mean"
        ),
        Mean_Compressed_Parameters=(
            "Compressed_Parameters", "mean"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio", "mean"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor", "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds", "mean"
        ),
        N=(
            "Reconstruction_Error", "count"
        )
    )
    .sort_values(
        ["Rank_R1", "Rank_R2"]
    )
)


# ============================================================
# SUBJECT + TASK SUMMARY
# ============================================================

subject_task_summary = (
    success_df
    .groupby(
        ["Subject", "Task",
         "Rank", "Rank_R1", "Rank_R2"],
        as_index=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error", "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error", "std"
        ),
        Mean_Explained_Variance=(
            "Explained_Variance", "mean"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio", "mean"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor", "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds", "mean"
        ),
        N=(
            "Reconstruction_Error", "count"
        )
    )
)


# ============================================================
# TASK SUMMARY
# ============================================================

task_summary = (
    success_df
    .groupby(
        ["Task", "Rank", "Rank_R1", "Rank_R2"],
        as_index=False
    )
    .agg(
        Mean_Reconstruction_Error=(
            "Reconstruction_Error", "mean"
        ),
        Std_Reconstruction_Error=(
            "Reconstruction_Error", "std"
        ),
        Mean_Explained_Variance=(
            "Explained_Variance", "mean"
        ),
        Mean_Compression_Ratio=(
            "Compression_Ratio", "mean"
        ),
        Mean_Compression_Factor=(
            "Compression_Factor", "mean"
        ),
        Mean_Runtime_Seconds=(
            "Runtime_Seconds", "mean"
        ),
        N=(
            "Reconstruction_Error", "count"
        )
    )
)


# ============================================================
# WRITE EXCEL
# ============================================================

excel_path = RESULTS_ROOT / "TT_rank_selection.xlsx"

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="TT_Metrics",
        index=False
    )

    overall_summary.to_excel(
        writer,
        sheet_name="Overall_Summary",
        index=False
    )

    subject_task_summary.to_excel(
        writer,
        sheet_name="Subject_Task_Summary",
        index=False
    )

    task_summary.to_excel(
        writer,
        sheet_name="Task_Summary",
        index=False
    )

print(f"\nSaved Excel workbook to:")
print(excel_path)


# ============================================================
# PLOTS
# ============================================================

# ------------------------------------------------------------
# 1. RECONSTRUCTION ERROR VS RANK
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

for _, row in overall_summary.iterrows():

    plt.scatter(
        row["Rank_R1"] * row["Rank_R2"],
        row["Mean_Reconstruction_Error"]
    )

    plt.annotate(
        f"({row['Rank_R1']},{row['Rank_R2']})",
        (
            row["Rank_R1"] * row["Rank_R2"],
            row["Mean_Reconstruction_Error"]
        ),
        fontsize=8
    )

plt.xlabel("TT rank product (r1 × r2)")
plt.ylabel("Mean reconstruction error")
plt.title("TT reconstruction error vs rank complexity")
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT / "01_reconstruction_error_vs_rank_complexity.png",
    dpi=300
)

plt.close()


# ------------------------------------------------------------
# 2. EXPLAINED VARIANCE VS RANK COMPLEXITY
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.scatter(
    overall_summary["Rank_R1"] *
    overall_summary["Rank_R2"],
    overall_summary["Mean_Explained_Variance_Percent"]
)

for _, row in overall_summary.iterrows():

    plt.annotate(
        f"({row['Rank_R1']},{row['Rank_R2']})",
        (
            row["Rank_R1"] * row["Rank_R2"],
            row["Mean_Explained_Variance_Percent"]
        ),
        fontsize=8
    )

plt.xlabel("TT rank product (r1 × r2)")
plt.ylabel("Mean explained variance (%)")
plt.title("TT explained variance vs rank complexity")
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT / "02_explained_variance_vs_rank_complexity.png",
    dpi=300
)

plt.close()


# ------------------------------------------------------------
# 3. RECONSTRUCTION ERROR VS COMPRESSION FACTOR
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.scatter(
    overall_summary["Mean_Compression_Factor"],
    overall_summary["Mean_Reconstruction_Error"]
)

for _, row in overall_summary.iterrows():

    plt.annotate(
        f"({row['Rank_R1']},{row['Rank_R2']})",
        (
            row["Mean_Compression_Factor"],
            row["Mean_Reconstruction_Error"]
        ),
        fontsize=8
    )

plt.xlabel("Compression factor")
plt.ylabel("Mean reconstruction error")
plt.title("TT compression–reconstruction trade-off")
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT / "03_compression_vs_reconstruction.png",
    dpi=300
)

plt.close()


# ------------------------------------------------------------
# 4. RECONSTRUCTION ERROR BY TASK
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

for task in sorted(success_df["Task"].unique()):

    task_data = (
        success_df[
            success_df["Task"] == task
        ]
        .groupby(
            ["Rank_R1", "Rank_R2"]
        )["Reconstruction_Error"]
        .mean()
        .reset_index()
    )

    task_data["complexity"] = (
        task_data["Rank_R1"] *
        task_data["Rank_R2"]
    )

    task_data = task_data.sort_values("complexity")

    plt.plot(
        task_data["complexity"],
        task_data["Reconstruction_Error"],
        marker="o",
        label=task
    )

plt.xlabel("TT rank product (r1 × r2)")
plt.ylabel("Mean reconstruction error")
plt.title("TT reconstruction error by task")
plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT / "04_reconstruction_error_by_task.png",
    dpi=300
)

plt.close()


# ------------------------------------------------------------
# 5. RUNTIME VS RANK COMPLEXITY
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.scatter(
    overall_summary["Rank_R1"] *
    overall_summary["Rank_R2"],
    overall_summary["Mean_Runtime_Seconds"]
)

for _, row in overall_summary.iterrows():

    plt.annotate(
        f"({row['Rank_R1']},{row['Rank_R2']})",
        (
            row["Rank_R1"] * row["Rank_R2"],
            row["Mean_Runtime_Seconds"]
        ),
        fontsize=8
    )

plt.xlabel("TT rank product (r1 × r2)")
plt.ylabel("Mean runtime (seconds)")
plt.title("TT computational cost vs rank complexity")
plt.grid(True)

plt.tight_layout()

plt.savefig(
    RESULTS_ROOT / "05_runtime_vs_rank_complexity.png",
    dpi=300
)

plt.close()


print("\n========================================")
print("TT RANK SELECTION COMPLETE")
print("========================================")
print(f"Results folder: {RESULTS_ROOT}")
print(f"Excel: {excel_path}")
print(f"CSV: {csv_path}")
print(f"Successful decompositions: {len(success_df)}")
print(f"Failed decompositions: {len(results_df) - len(success_df)}")

Found 32 tensor files

[1/32] 002/auditory/run01_epochs.npz
Tensor shape: (200, 30, 1401)
  TT ranks = (1, 3, 3, 1)
  TT ranks = (1, 3, 5, 1)
  TT ranks = (1, 5, 3, 1)
  TT ranks = (1, 5, 5, 1)
  TT ranks = (1, 5, 7, 1)
  TT ranks = (1, 7, 5, 1)
  TT ranks = (1, 7, 7, 1)
  TT ranks = (1, 7, 10, 1)
  TT ranks = (1, 10, 7, 1)
  TT ranks = (1, 10, 10, 1)
  TT ranks = (1, 10, 15, 1)
  TT ranks = (1, 15, 10, 1)
  TT ranks = (1, 15, 15, 1)
  TT ranks = (1, 15, 20, 1)
  TT ranks = (1, 20, 15, 1)
  TT ranks = (1, 20, 20, 1)

[2/32] 002/auditory/run02_epochs.npz
Tensor shape: (200, 30, 1401)
  TT ranks = (1, 3, 3, 1)
  TT ranks = (1, 3, 5, 1)
  TT ranks = (1, 5, 3, 1)
  TT ranks = (1, 5, 5, 1)
  TT ranks = (1, 5, 7, 1)
  TT ranks = (1, 7, 5, 1)
  TT ranks = (1, 7, 7, 1)
  TT ranks = (1, 7, 10, 1)
  TT ranks = (1, 10, 7, 1)
  TT ranks = (1, 10, 10, 1)
  TT ranks = (1, 10, 15, 1)
  TT ranks = (1, 15, 10, 1)
  TT ranks = (1, 15, 15, 1)
  TT ranks = (1, 15, 20, 1)
  TT ranks = (1, 20, 15, 1)
  TT r